# EDA: Label Distribution by Dataset ("pole")

This notebook follows the same structure as `eda_labels.ipynb`, but computes and displays label statistics **separately for each dataset** under `dataset/Poles2025`.

For each dataset, it reports:
- Number of labeled objects by split
- Distribution summary for `x_center`, `y_center`, `width`, `height`, and corners
- Min / max / average pole position in frame
- Per-label corner statistics
- Dataset-specific visualizations

In [1]:
from pathlib import Path
import re
import yaml
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT_DIR = Path(".")
DATASET_ROOT = ROOT_DIR / "dataset" / "Poles2025"

pd.set_option("display.max_columns", 120)

print(f"Dataset root: {DATASET_ROOT.resolve()}")

Dataset root: /home/chongy/stuff/codespace/snowpole-detection-ad/dataset/Poles2025


In [2]:
def parse_names_field(names_field):
    if isinstance(names_field, dict):
        return {int(k): str(v) for k, v in names_field.items()}
    if isinstance(names_field, list):
        return {i: str(v) for i, v in enumerate(names_field)}
    return {}


def infer_split_from_path(path: Path):
    parts = [p.lower() for p in path.parts]
    if "train" in parts:
        return "train"
    if "validation" in parts or "val" in parts or "valid" in parts:
        return "valid"
    if "test" in parts:
        return "test"
    return "unknown"


def looks_like_yolo_label_file(path: Path):
    if path.suffix.lower() != ".txt":
        return False
    if path.name.lower() in {"train.txt", "validation.txt", "val.txt", "test.txt"}:
        return False

    try:
        lines = [ln.strip() for ln in path.read_text().splitlines() if ln.strip()]
    except Exception:
        return False

    if not lines:
        return False

    sample = lines[: min(5, len(lines))]
    pattern = re.compile(r"^-?\d+\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?\s+[-+]?\d*\.?\d+(?:[eE][-+]?\d+)?(?:\s+.*)?$")
    return all(pattern.match(ln) for ln in sample)


def read_yolo_labels(dataset_dir: Path):
    data_yaml = dataset_dir / "data.yaml"
    if not data_yaml.exists():
        return pd.DataFrame(), {}

    meta = yaml.safe_load(data_yaml.read_text()) or {}
    class_map = parse_names_field(meta.get("names", {}))

    rows = []
    for txt in dataset_dir.rglob("*.txt"):
        if not looks_like_yolo_label_file(txt):
            continue

        rel = txt.relative_to(dataset_dir)
        split = infer_split_from_path(rel)

        for line_no, line in enumerate(txt.read_text().splitlines(), start=1):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) < 5:
                continue
            try:
                cls_id = int(float(parts[0]))
                x_center, y_center, width, height = map(float, parts[1:5])
            except ValueError:
                continue

            rows.append(
                {
                    "dataset": dataset_dir.name,
                    "split": split,
                    "label_file": str(rel),
                    "line_no": line_no,
                    "class_id": cls_id,
                    "class_name": class_map.get(cls_id, str(cls_id)),
                    "x_center": x_center,
                    "y_center": y_center,
                    "width": width,
                    "height": height,
                    "area": width * height,
                }
            )

    return pd.DataFrame(rows), class_map


def collect_all_labels(dataset_root: Path):
    dfs = []
    class_maps = {}

    for data_yaml in dataset_root.rglob("data.yaml"):
        dataset_dir = data_yaml.parent
        df_ds, cmap = read_yolo_labels(dataset_dir)
        if not df_ds.empty:
            dfs.append(df_ds)
            class_maps[dataset_dir.name] = cmap

    if not dfs:
        return pd.DataFrame(), class_maps

    return pd.concat(dfs, ignore_index=True), class_maps

In [3]:
df, class_maps = collect_all_labels(DATASET_ROOT)

if df.empty:
    raise RuntimeError("No YOLO label annotations found under dataset/Poles2025.")

# Compute bounding-box corners from YOLO format (normalized)
df = df.copy()
df["x_min"] = (df["x_center"] - df["width"] / 2).clip(0, 1)
df["y_min"] = (df["y_center"] - df["height"] / 2).clip(0, 1)
df["x_max"] = (df["x_center"] + df["width"] / 2).clip(0, 1)
df["y_max"] = (df["y_center"] + df["height"] / 2).clip(0, 1)

# Focus on poles (case-insensitive). Falls back to all rows if class names are numeric.
is_pole = df["class_name"].astype(str).str.lower().eq("pole")
pole_df = df[is_pole].copy()
if pole_df.empty:
    pole_df = df.copy()

datasets = sorted(df["dataset"].unique().tolist())

print("Detected datasets:", datasets)
print("Total labeled objects (all classes):", len(df))
print("Total rows used for pole analysis:", len(pole_df))
print("Class maps:")
for ds, cmap in class_maps.items():
    print(f"  - {ds}: {cmap}")

Detected datasets: ['Road_poles_iPhone', 'roadpoles_v1']
Total labeled objects (all classes): 2004
Total rows used for pole analysis: 2004
Class maps:
  - Road_poles_iPhone: {0: 'pole'}
  - roadpoles_v1: {0: 'pole'}


In [4]:
for dataset_name in sorted(pole_df["dataset"].unique()):
    print("\n" + "=" * 90)
    print(f"DATASET: {dataset_name}")
    print("=" * 90)

    ds_pole = pole_df[pole_df["dataset"] == dataset_name].copy()
    ds_all = df[df["dataset"] == dataset_name].copy()

    print(f"Rows used for pole analysis in dataset: {len(ds_pole):,}")

    # Count distribution by split
    count_by_split = (
        ds_pole.groupby("split", dropna=False)
        .size()
        .rename("n_boxes")
        .reset_index()
        .sort_values("split")
    )
    print("\nCount by split:")
    display(count_by_split)

    # Position + size + corner stats (pole only)
    stats_cols = ["x_center", "y_center", "width", "height", "area", "x_min", "y_min", "x_max", "y_max"]
    summary = ds_pole[stats_cols].agg(["min", "max", "mean", "median", "std"]).T
    summary = summary.rename(columns={"mean": "avg"})
    print("\nPole summary stats (normalized):")
    display(summary)

    # Explicit position stats requested
    position_stats = pd.DataFrame(
        {
            "metric": ["x_center", "y_center"],
            "min": [ds_pole["x_center"].min(), ds_pole["y_center"].min()],
            "max": [ds_pole["x_center"].max(), ds_pole["y_center"].max()],
            "avg": [ds_pole["x_center"].mean(), ds_pole["y_center"].mean()],
        }
    )
    print("\nPole position in frame (normalized):")
    display(position_stats)

    # Per-label corner stats (all labels in this dataset)
    corner_cols = ["x_min", "y_min", "x_max", "y_max"]
    corner_stats_by_label = (
        ds_all.groupby("class_name", dropna=False)[corner_cols]
        .agg(["min", "max", "mean", "median", "std"])
    )
    print("\nBounding-box corner stats by label (normalized):")
    display(corner_stats_by_label)

    per_split_position = (
        ds_pole.groupby("split")[["x_center", "y_center"]]
        .agg(["min", "max", "mean"])
    )
    print("\nPer-split position summary:")
    display(per_split_position)


DATASET: Road_poles_iPhone
Rows used for pole analysis in dataset: 1,499

Count by split:


,split,n_boxes
0,train,1169
1,valid,330



Pole summary stats (normalized):


,min,max,avg,median,std
x_center,0.005394,0.987032,0.574943,0.726565,0.320974
y_center,0.475680,0.610956,0.559090,0.560120,0.017179
width,0.002769,0.074713,0.017891,0.015926,0.010477
height,0.015812,0.180880,0.057265,0.051161,0.024503
area,0.000075,0.010007,0.001089,0.000774,0.000990
x_min,0.000000,0.979629,0.565998,0.719009,0.319935
y_min,0.439437,0.572917,0.530457,0.532636,0.016871
x_max,0.010788,1.000000,0.583889,0.734445,0.322095
y_max,0.505146,0.685516,0.587722,0.584958,0.024613



Pole position in frame (normalized):


,metric,min,max,avg
0,x_center,0.005394,0.987032,0.574943
1,y_center,0.475680,0.610956,0.559090


... (10 outputs left out)


Bounding-box corner stats by label (normalized):


x_min                                             y_min  \
                 min       max      mean    median       std       min   
class_name                                                               
pole        0.000442  0.986719  0.455274  0.573714  0.243209  0.450157   

                                                      x_max            \
                 max      mean   median       std       min       max   
class_name                                                              
pole        0.644454  0.546934  0.54789  0.021031  0.006386  0.996563   

                                             y_max                      \
                mean    median       std       min       max      mean   
class_name                                                               
pole        0.464175  0.581145  0.246488  0.590347  0.943079  0.666705   

                                
              median       std  
class_name                      
pole        0.645348  0.065415


Per-split position summary:


x_center                      y_center                    
            min       max      mean       min       max      mean
split                                                            
train  0.003414  0.980495  0.460914  0.560389  0.730339  0.607254
valid  0.012852  0.991641  0.455599  0.562678  0.714528  0.605316

## Visualizations per dataset

Each dataset gets its own figures for quick comparison.

In [ ]:
for dataset_name in sorted(pole_df["dataset"].unique()):
    ds_pole = pole_df[pole_df["dataset"] == dataset_name].copy()

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle(f"{dataset_name} — Pole Label Distribution", fontsize=14, y=1.02)

    # 1) Count of boxes by split
    (
        ds_pole.groupby("split").size().reindex(["train", "valid", "test", "unknown"]).dropna()
        .plot(kind="bar", ax=axes[0, 0], title="Pole boxes per split")
    )
    axes[0, 0].set_ylabel("count")
    axes[0, 0].tick_params(axis="x", rotation=0)

    # 2) Histograms of center positions
    axes[0, 1].hist(ds_pole["x_center"], bins=30, alpha=0.7, label="x_center")
    axes[0, 1].hist(ds_pole["y_center"], bins=30, alpha=0.7, label="y_center")
    axes[0, 1].set_title("Distribution of pole center positions")
    axes[0, 1].set_xlabel("normalized coordinate")
    axes[0, 1].legend()

    # 3) Spatial distribution scatter
    sample_scatter = ds_pole.sample(min(5000, len(ds_pole)), random_state=42)
    axes[1, 0].scatter(sample_scatter["x_center"], sample_scatter["y_center"], s=8, alpha=0.3)
    axes[1, 0].set_title("Pole center positions")
    axes[1, 0].set_xlabel("x_center")
    axes[1, 0].set_ylabel("y_center")
    axes[1, 0].set_xlim(0, 1)
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].invert_yaxis()

    # 4) Box size distribution
    axes[1, 1].hist(ds_pole["area"], bins=35, alpha=0.8, color="#2ca02c")
    axes[1, 1].set_title("Pole bbox area distribution")
    axes[1, 1].set_xlabel("area (normalized)")

    plt.tight_layout()
    plt.show()

In [ ]:
for dataset_name in sorted(pole_df["dataset"].unique()):
    ds_pole = pole_df[pole_df["dataset"] == dataset_name].copy()

    # Corner distributions
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"{dataset_name} — Corner Diagnostics", fontsize=14, y=1.02)

    axes[0, 0].hist(ds_pole["x_min"], bins=35, alpha=0.8, color="#1f77b4")
    axes[0, 0].set_title("x_min distribution")
    axes[0, 0].set_xlabel("x_min")

    axes[0, 1].hist(ds_pole["x_max"], bins=35, alpha=0.8, color="#ff7f0e")
    axes[0, 1].set_title("x_max distribution")
    axes[0, 1].set_xlabel("x_max")

    axes[0, 2].hist(ds_pole["x_max"] - ds_pole["x_min"], bins=35, alpha=0.8, color="#2ca02c")
    axes[0, 2].set_title("bbox width from corners")
    axes[0, 2].set_xlabel("x_max - x_min")

    axes[1, 0].hist(ds_pole["y_min"], bins=35, alpha=0.8, color="#9467bd")
    axes[1, 0].set_title("y_min distribution")
    axes[1, 0].set_xlabel("y_min")

    axes[1, 1].hist(ds_pole["y_max"], bins=35, alpha=0.8, color="#8c564b")
    axes[1, 1].set_title("y_max distribution")
    axes[1, 1].set_xlabel("y_max")

    axes[1, 2].hist(ds_pole["y_max"] - ds_pole["y_min"], bins=35, alpha=0.8, color="#e377c2")
    axes[1, 2].set_title("bbox height from corners")
    axes[1, 2].set_xlabel("y_max - y_min")

    for ax in axes.flat:
        ax.set_xlim(left=0)

    plt.tight_layout()
    plt.show()

    # Spatial corner maps
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle(f"{dataset_name} — Corner Spatial Map", fontsize=14, y=1.03)

    sample = ds_pole.sample(min(4000, len(ds_pole)), random_state=42)

    axes[0].scatter(sample["x_min"], sample["y_min"], s=8, alpha=0.25, color="#1f77b4")
    axes[0].set_title("Lower-left corner (x_min, y_min)")
    axes[0].set_xlabel("x_min")
    axes[0].set_ylabel("y_min")
    axes[0].set_xlim(0, 1)
    axes[0].set_ylim(0, 1)
    axes[0].invert_yaxis()

    axes[1].scatter(sample["x_max"], sample["y_max"], s=8, alpha=0.25, color="#ff7f0e")
    axes[1].set_title("Upper-right corner (x_max, y_max)")
    axes[1].set_xlabel("x_max")
    axes[1].set_ylabel("y_max")
    axes[1].set_xlim(0, 1)
    axes[1].set_ylim(0, 1)
    axes[1].invert_yaxis()

    plt.tight_layout()
    plt.show()